In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, NotRequired
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from pprint import pprint

In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [8]:
class BlogState(TypedDict):
    title: str
    outline: NotRequired[str]
    content: NotRequired[str]

In [5]:
def generate_outline(state: BlogState) -> BlogState:
    title = state.get('title')

    prompt = f'Generate outline for the following: {title}'

    outline = str(model.invoke(prompt).content)

    state['outline'] = outline

    return state

In [6]:
def generate_blog(state: BlogState) -> BlogState:
    outline = state.get('outline')

    prompt = f'Generate a detailed blog post for the title outline: {outline}'

    content = str(model.invoke(prompt).content)

    state['content'] = content

    return state

In [7]:
graph = StateGraph(BlogState)

graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)

graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', END)

workflow = graph.compile()

In [9]:
initial_state: BlogState = {'title': 'F1 vs Football'}
final_state = workflow.invoke(initial_state)

In [12]:
pprint(final_state)

{'content': '**The High-Octane World of F1 vs The Beautiful Game of Football: '
            'A Comprehensive Comparison**\n'
            '\n'
            'The world of sports is home to numerous thrilling disciplines, '
            'but two of the most popular and captivating ones are Formula 1 '
            '(F1) and Football. With millions of fans worldwide, both sports '
            'have become an integral part of modern entertainment, captivating '
            'audiences with their unique blend of speed, skill, and strategy. '
            'While F1 and Football may seem like vastly different sports on '
            'the surface, they share some intriguing similarities, but also '
            'have many differences in terms of their fan base, financial '
            "aspects, and competitive nature. In this article, we'll delve "
            'into the fascinating world of F1 and Football, exploring their '
            'similarities and differences, and examining what makes each spo